# M5.S1 — Hybrid computing patterns: MPI + OpenMP + GPU
**HPC 2026–27 · Syllabus Session 21 · SciTech teaching notebook · Build M5S1-2026-10-13-v1**

**Predict → inspect real code → run on allocated hardware → verify → change one setting → explain.**

### The scientific problem
A hot edge heats a metal plate. At each time step, an interior cell takes the average of four neighbours. We start with one serial CPU process, split the grid into MPI subdomains, use OpenMP threads inside the ranks, and examine an **optional** GPU kernel.

**What this notebook actually executes:** a real C MPI+OpenMP heat solver with halo exchanges and a small MPI+OpenMP rank/thread mapper in **one Slurm CPU batch job**. Optional single-GPU CUDA vector addition uses a **separate** GPU Slurm job, only if you opt in. The CUDA example illustrates acceleration and transfer overhead; it is **not** a multi-GPU MPI application.

| Lecture slides | Practical activity |
|---|---|
| 2–6 | Predict the required resources |
| 7–11 | Inspect heat code, observe MPI halo exchanges and OpenMP loop |
| 12, 19 | Inspect physical allocation, rank/thread map and placement |
| 14–17 | Optional real GPU kernel, distinguish GPU compute from transfers |
| 20–21 | Verify equal numerical solutions; compare actual elapsed times |
| 22–25 | Make one controlled change and explain the outcome |

**Validation labels:** PASS = required checks passed; FAIL = an observed error or numerical mismatch; PENDING = submitted job not finished; NOT RUN = execution not attempted; SKIPPED = optional capability unavailable or not requested. Timings are produced by your real job, not stored example numbers.

**Precondition:** open from GitHub in [SciTech JupyterHub](https://jupyter.iesci.tech:8085/) using **File → Open from URL**, and save a personal copy. The work directory must be user-owned and visible from the compute node. Do not execute MPI batches on the login node. See the [cluster guide](https://github.com/OscarDiez/hpc_course/blob/main/cluster/README.md).

## 0 — What should happen?
**Situation:** A 2D heat simulation uses a grid of local temperatures.

**Predict before running:**
1. Can two independent MPI processes update different grid rows?
2. Why do they need to exchange halo rows each step?
3. Would 2 MPI ranks × 2 OpenMP threads print 2 or 4 rank/thread pairs?
4. Must MPI+OpenMP be faster than serial for a small grid?

**Expected insight:** correct decomposition requires halo exchange; 2×2 creates four rank/thread pairs; speedup depends on size and overhead. Record your predictions, then compare them to real output.

**Notebook route:** run **setup → materialize source → submit CPU job → collect → inspect results → optional GPU → change one setting**. Do not rerun submission while the same job is pending.

## 1 — Settings and environment
The default CPU lab uses **one node**, **two MPI ranks** and **at most two OpenMP threads per rank**. Three versions of the same numerical problem run sequentially in the allocated job. The default grid is small enough for classroom use.

Only change `GRID_N` or `STEPS` at first. Larger jobs may change timings, but communication and synchronization remain visible even in small examples. MPI ranks are logical processes; on one node they still have separate address spaces.

In [ ]:
from pathlib import Path
import os, sys, json, re, time, uuid, hashlib, shutil, subprocess, statistics
from IPython.display import display, Markdown
BUILD = "M5S1-2026-10-13-v1"
WORKSPACE = (Path.cwd() / "m5s1_runs").resolve()  # MUST be a student-owned filesystem shared with CPU compute nodes
GRID_N = 256                # 16..2048; 256 is a quick classroom problem
STEPS = 350                # 1..10000; diffusion time steps
NEW_RUN = False            # True = submit a new job even with identical settings
RUN_GPU = False            # Optional: requests a GPU job only when explicitly changed to True
GPU_NEW_RUN = False
WORKSPACE.mkdir(parents=True, exist_ok=True)
assert 16 <= GRID_N <= 2048 and 1 <= STEPS <= 10000
print("Build:", BUILD, "| Python host:", os.uname().nodename)
print("Working directory:", WORKSPACE)
for tool in ("sbatch", "squeue", "sinfo", "srun", "mpicc"):
    print(f"{tool}: {shutil.which(tool) or 'not visible in notebook kernel'}")
print("Slurm allocation for notebook:", os.environ.get("SLURM_JOB_ID", "not set"))
print("CPU memory/count seen by notebook is NOT the resource allocation for the later batch job.")
print("GPU job:", "opt-in" if RUN_GPU else "SKIPPED (RUN_GPU=False)")

## 2 — Read the actual C code
The following cell **embeds all sources**, so students do not have to download other files or install packages. The strings are written unchanged into each run directory.

**Inspect the executed operations:**
- `MPI_Sendrecv` exchanges an owned boundary row with a neighbouring rank's halo.
- `#pragma omp parallel for` splits locally owned rows among CPU threads.
- `old[]` is read during a time step; `next[]` is written, then the arrays swap.
- `MPI_Gatherv` collects the completed temperature field.
- `MPI_Reduce` aggregates the checksum and slowest-rank elapsed time.

**Scientific boundary condition:** the top plate edge remains at 100 °C, other edges at 0 °C; the interior starts at 0 °C. This is a teaching discretisation of diffusion, not a validated engineering heat-transfer model.

In [ ]:
SRC_HELLO = "#include <mpi.h>\n#include <omp.h>\n#include <stdio.h>\nint main(int argc,char **argv) {\n  int provided,rank,size;\n  MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);\n  MPI_Comm_rank(MPI_COMM_WORLD,&rank);\n  MPI_Comm_size(MPI_COMM_WORLD,&size);\n  #pragma omp parallel\n  {\n    #pragma omp critical\n    printf(\"MAP rank=%d of=%d thread=%d of_threads=%d\\n\",\n           rank,size,omp_get_thread_num(),omp_get_num_threads());\n  }\n  MPI_Finalize(); return 0;\n}"
SRC_HEAT = "/* Explicit 5-point Jacobi heat update. Row decomposition and MPI halos.\n   At each step every rank reads old[] and writes next[]; no in-place race. */\n#include <mpi.h>\n#include <omp.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\nstatic int rows_for(int r,int n,int p){return n/p+(r<n%p);}\nstatic int offset_for(int r,int n,int p){return r*(n/p)+(r<n%p?r:n%p);}\nint main(int argc,char **argv) {\n  int rank,p,provided;\n  MPI_Init_thread(&argc,&argv,MPI_THREAD_FUNNELED,&provided);\n  MPI_Comm_rank(MPI_COMM_WORLD,&rank); MPI_Comm_size(MPI_COMM_WORLD,&p);\n  if(argc!=5){if(rank==0)fprintf(stderr,\"usage: heat N STEPS output.bin label\\n\"); MPI_Abort(MPI_COMM_WORLD,2);}\n  int n=atoi(argv[1]),steps=atoi(argv[2]),rows=rows_for(rank,n,p),first=offset_for(rank,n,p);\n  if(n<16||n>2048||steps<1||steps>10000||p>n){if(rank==0)fprintf(stderr,\"invalid parameters\\n\"); MPI_Abort(MPI_COMM_WORLD,3);}\n  size_t count=(size_t)(rows+2)*n;\n  double *old=(double*)calloc(count,sizeof(double)),*next=(double*)calloc(count,sizeof(double));\n  if(!old||!next)MPI_Abort(MPI_COMM_WORLD,4);\n  for(int y=1;y<=rows;y++)if(first+y-1==0)for(int x=0;x<n;x++)old[(size_t)y*n+x]=100.0;\n  int up=(rank==0?MPI_PROC_NULL:rank-1),down=(rank==p-1?MPI_PROC_NULL:rank+1);\n  MPI_Barrier(MPI_COMM_WORLD); double t0=MPI_Wtime();\n  for(int step=0;step<steps;step++){\n    MPI_Sendrecv(old+n,n,MPI_DOUBLE,up,10,old+(size_t)(rows+1)*n,n,MPI_DOUBLE,down,10,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    MPI_Sendrecv(old+(size_t)rows*n,n,MPI_DOUBLE,down,11,old,n,MPI_DOUBLE,up,11,MPI_COMM_WORLD,MPI_STATUS_IGNORE);\n    #pragma omp parallel for schedule(static)\n    for(int y=1;y<=rows;y++){\n      int global_y=first+y-1;\n      if(global_y==0||global_y==n-1){\n        memcpy(next+(size_t)y*n,old+(size_t)y*n,(size_t)n*sizeof(double));\n      }else{\n        next[(size_t)y*n]=0.0; next[(size_t)y*n+n-1]=0.0;\n        for(int x=1;x<n-1;x++)next[(size_t)y*n+x]=0.25*(old[(size_t)(y-1)*n+x]+old[(size_t)(y+1)*n+x]+old[(size_t)y*n+x-1]+old[(size_t)y*n+x+1]);\n      }\n    }\n    double *tmp=old;old=next;next=tmp;\n  }\n  MPI_Barrier(MPI_COMM_WORLD); double elapsed=MPI_Wtime()-t0,max_elapsed=0;\n  MPI_Reduce(&elapsed,&max_elapsed,1,MPI_DOUBLE,MPI_MAX,0,MPI_COMM_WORLD);\n  double partial=0,probe_local=0,checksum=0,probe=0;\n  for(int y=1;y<=rows;y++){\n    int gy=first+y-1;\n    for(int x=0;x<n;x++)partial+=old[(size_t)y*n+x];\n    if(gy==8)probe_local=old[(size_t)y*n+n/2];\n  }\n  MPI_Reduce(&partial,&checksum,1,MPI_DOUBLE,MPI_SUM,0,MPI_COMM_WORLD);\n  MPI_Reduce(&probe_local,&probe,1,MPI_DOUBLE,MPI_SUM,0,MPI_COMM_WORLD);\n  double *global=NULL;int *counts=NULL,*offsets=NULL;\n  if(rank==0){\n    global=(double*)malloc((size_t)n*n*sizeof(double));\n    counts=(int*)malloc((size_t)p*sizeof(int));offsets=(int*)malloc((size_t)p*sizeof(int));\n    if(!global||!counts||!offsets)MPI_Abort(MPI_COMM_WORLD,5);\n    for(int r=0;r<p;r++){counts[r]=rows_for(r,n,p)*n;offsets[r]=offset_for(r,n,p)*n;}\n  }\n  MPI_Gatherv(old+n,rows*n,MPI_DOUBLE,global,counts,offsets,MPI_DOUBLE,0,MPI_COMM_WORLD);\n  if(rank==0){\n    FILE *f=fopen(argv[3],\"wb\");\n    if(!f || fwrite(global,sizeof(double),(size_t)n*n,f)!=(size_t)n*n){fprintf(stderr,\"Unable to write heat grid\\n\");MPI_Abort(MPI_COMM_WORLD,6);}\n    fclose(f);\n    printf(\"RESULT mode=%s ranks=%d threads=%d n=%d steps=%d elapsed_s=%.9f checksum=%.12f probe=%.12f\\n\",\n      argv[4],p,omp_get_max_threads(),n,steps,max_elapsed,checksum,probe);\n    fflush(stdout);\n  }\n  free(old);free(next);free(global);free(counts);free(offsets);\n  MPI_Finalize();return 0;\n}"
SRC_CUDA = "/* Single-GPU teaching experiment: one vector addition, actual CUDA events. */\n#include <cuda_runtime.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <math.h>\n#include <chrono>\n#define CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,\"CUDA error: %s\\n\",cudaGetErrorString(e));return 2;} }while(0)\n__global__ void add(const float *a,const float *b,float *c,int n){int i=blockIdx.x*blockDim.x+threadIdx.x;if(i<n)c[i]=a[i]+b[i];}\nint main(){\n  const int n=1<<20; size_t bytes=(size_t)n*sizeof(float);\n  float *a=(float*)malloc(bytes),*b=(float*)malloc(bytes),*c=(float*)malloc(bytes);\n  if(!a||!b||!c)return 3;\n  for(int i=0;i<n;i++){a[i]=0.25f*i;b[i]=2.0f;}\n  float *da=NULL,*db=NULL,*dc=NULL;\n  auto start=std::chrono::steady_clock::now();\n  CHECK(cudaMalloc((void**)&da,bytes));CHECK(cudaMalloc((void**)&db,bytes));CHECK(cudaMalloc((void**)&dc,bytes));\n  CHECK(cudaMemcpy(da,a,bytes,cudaMemcpyHostToDevice));CHECK(cudaMemcpy(db,b,bytes,cudaMemcpyHostToDevice));\n  cudaEvent_t e0,e1;CHECK(cudaEventCreate(&e0));CHECK(cudaEventCreate(&e1));\n  CHECK(cudaEventRecord(e0));\n  add<<<(n+255)/256,256>>>(da,db,dc,n);\n  CHECK(cudaGetLastError());CHECK(cudaEventRecord(e1));CHECK(cudaEventSynchronize(e1));\n  float kernel_ms=0;CHECK(cudaEventElapsedTime(&kernel_ms,e0,e1));\n  CHECK(cudaMemcpy(c,dc,bytes,cudaMemcpyDeviceToHost));\n  auto end=std::chrono::steady_clock::now();\n  double total_ms=std::chrono::duration<double,std::milli>(end-start).count();\n  float worst=0;\n  for(int i=0;i<n;i++){float diff=fabsf(c[i]-(a[i]+b[i]));if(diff>worst)worst=diff;}\n  printf(\"GPU_RESULT n=%d kernel_ms=%.5f end_to_end_ms=%.5f max_error=%.7g\\n\",n,kernel_ms,total_ms,worst);\n  CHECK(cudaEventDestroy(e0));CHECK(cudaEventDestroy(e1));\n  CHECK(cudaFree(da));CHECK(cudaFree(db));CHECK(cudaFree(dc));\n  free(a);free(b);free(c);return worst<0.001f?0:4;\n}"
CPU_SCRIPT = "#!/bin/bash\n#SBATCH --job-name=m5s1-hybrid\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=2\n#SBATCH --cpus-per-task=2\n#SBATCH --mem=2G\n#SBATCH --time=00:06:00\n#SBATCH --output=cpu-%j.out\nset -euo pipefail\ncd \"$SLURM_SUBMIT_DIR\"\nif [ -f /etc/profile.d/modules.sh ]; then source /etc/profile.d/modules.sh; fi\nif command -v module >/dev/null 2>&1; then\n  module load OpenMPI/4.1.5-GCC-12.3.0 2>/dev/null || module load foss/2023b 2>/dev/null || true\n  module list 2>&1 || true\nfi\ncommand -v mpicc || { echo 'FAIL: mpicc unavailable'; exit 3; }\ncommand -v srun || { echo 'FAIL: srun unavailable'; exit 4; }\necho \"JOB allocation=$SLURM_JOB_ID host=$(hostname) nodes=$SLURM_JOB_NUM_NODES\"\nmpicc -O2 -fopenmp hybrid_hello.c -o hybrid_hello\nmpicc -O2 -fopenmp hybrid_heat.c -o hybrid_heat\nexport OMP_DYNAMIC=FALSE\nexport OMP_NUM_THREADS=2\necho '--- Rank / thread map (real Slurm allocation) ---'\nsrun --ntasks=2 --cpus-per-task=2 ./hybrid_hello\necho '--- Same scientific problem, three configurations ---'\nexport OMP_NUM_THREADS=1\nsrun --ntasks=1 --cpus-per-task=1 ./hybrid_heat __N__ __STEPS__ serial.bin serial\nsrun --ntasks=2 --cpus-per-task=1 ./hybrid_heat __N__ __STEPS__ mpi.bin mpi\nexport OMP_NUM_THREADS=2\nsrun --ntasks=2 --cpus-per-task=2 ./hybrid_heat __N__ __STEPS__ hybrid.bin hybrid\necho 'CPU_LAB_DONE'\n"
GPU_SCRIPT = "#!/bin/bash\n#SBATCH --job-name=m5s1-gpu\n#SBATCH --partition=gpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=2\n#SBATCH --gpus=1\n#SBATCH --mem=2G\n#SBATCH --time=00:05:00\n#SBATCH --output=gpu-%j.out\nset -euo pipefail\ncd \"$SLURM_SUBMIT_DIR\"\nif [ -f /etc/profile.d/modules.sh ]; then source /etc/profile.d/modules.sh; fi\nif command -v module >/dev/null 2>&1; then\n  module load nvhpc/25.7 2>/dev/null || true\n  module list 2>&1 || true\nfi\ncommand -v nvidia-smi || { echo 'SKIPPED: no NVIDIA runtime'; exit 0; }\nnvidia-smi --query-gpu=name --format=csv,noheader\ncommand -v nvcc || { echo 'SKIPPED: nvcc unavailable; see M2.S4 GPU notebook'; exit 0; }\nnvcc -O2 gpu_vector_add.cu -o gpu_vector_add\n./gpu_vector_add\necho 'GPU_LAB_DONE'\n"
SOURCES = {"hybrid_hello.c": SRC_HELLO, "hybrid_heat.c": SRC_HEAT, "gpu_vector_add.cu": SRC_CUDA}
print("Actual embedded source files:", ", ".join(f"{k} ({len(v)} bytes)" for k,v in SOURCES.items()))
print("The heat loop uses MPI_Sendrecv and #pragma omp parallel for:", "MPI_Sendrecv" in SRC_HEAT, "#pragma omp parallel for" in SRC_HEAT)
print("\n--- Executed halo exchange ---")
i = SRC_HEAT.index("MPI_Sendrecv")
print(SRC_HEAT[i:i+380])
print("\n--- Executed OpenMP loop ---")
i = SRC_HEAT.index("#pragma omp parallel for")
print(SRC_HEAT[i:i+620])

## 3 — Materialize an immutable, reproducible run
Files are created in a unique folder determined by the source and scientific settings. If the settings remain unchanged and `NEW_RUN=False`, the notebook reuses that run rather than silently submitting again.

**Why this matters:** a job may be queued while you edit the notebook. Its frozen source and settings must remain unchanged. Each job records its own Slurm stdout log.

The program runs inside Slurm with the supported SciTech MPI module `OpenMPI/4.1.5-GCC-12.3.0` (falling back to a compatible environment only when provided). Each scientific run writes a binary temperature grid that the notebook later verifies and plots.

In [ ]:
FINGERPRINT = hashlib.sha256(json.dumps({"build":BUILD,"n":GRID_N,"steps":STEPS,"sources":SOURCES,"script":CPU_SCRIPT},sort_keys=True).encode()).hexdigest()
RUN_DIR = WORKSPACE / ("cpu-"+FINGERPRINT[:18]+("-"+uuid.uuid4().hex[:6] if NEW_RUN else ""))
RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCES.items():
    target = RUN_DIR / name
    if not target.exists(): target.write_text(source)
    elif target.read_text() != source: raise RuntimeError("Frozen source mismatch: "+str(target))
batch = CPU_SCRIPT.replace("__N__",str(GRID_N)).replace("__STEPS__",str(STEPS))
for name,source in {"cpu.sbatch":batch}.items():
    target=RUN_DIR/name
    if not target.exists():target.write_text(source)
    elif target.read_text()!=source:raise RuntimeError("Frozen batch mismatch")
print("Frozen source/settings fingerprint:",FINGERPRINT[:18])
print("Run dir:",RUN_DIR)
print("Grid:",GRID_N,"x",GRID_N,"| steps:",STEPS)
print("Batch resources: 1 CPU node, 2 tasks, 2 cores/task, 2 GiB, 6 minutes")
print("Example rank layouts: 1x1 (serial); 2x1 (MPI); 2x2 (MPI+OpenMP)")

## 4 — Submit the real CPU job
**Predict:** How many `MAP` lines will the 2-rank × 2-thread experiment print? Which heat version will use the most MPI messages?

The batch job compiles the **real C source** on the allocated node and runs three cases, with **identical grid size and time steps**:
1. Serial: 1 rank × 1 thread.
2. MPI: 2 ranks × 1 thread.
3. MPI + OpenMP: 2 ranks × 2 threads.

We intentionally do not compare all three on identical CPU core counts. These comparisons show *architecture and correctness*; elapsed times also reflect unequal resource use, rank placement and small-problem overhead.

**No Slurm?** The cell reports **NOT RUN** rather than inventing a result. If your job is queued, rerun the **collection** cell later.

In [ ]:
CPU_JOB_FILE = RUN_DIR / "cpu_job_id.txt"
CPU_JOB_ID = CPU_JOB_FILE.read_text().strip() if CPU_JOB_FILE.exists() else None
if CPU_JOB_ID and not NEW_RUN:
    print("Existing CPU job:",CPU_JOB_ID,"(no resubmission)")
elif not shutil.which("sbatch"):
    print("NOT RUN: sbatch unavailable. Use SciTech JupyterHub / cluster environment.")
else:
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU","SLURM_MEM_PER_GPU","SLURM_MEM_PER_NODE"):env.pop(key,None)
    p = subprocess.run(["sbatch","--parsable","--chdir="+str(RUN_DIR),str(RUN_DIR/"cpu.sbatch")],
                       capture_output=True,text=True,env=env,timeout=35)
    if p.returncode:
        print("FAIL: CPU job not accepted:",p.stderr.strip() or p.stdout.strip())
    else:
        CPU_JOB_ID=p.stdout.strip().split(";")[0]
        CPU_JOB_FILE.write_text(CPU_JOB_ID)
        print("Submitted CPU job:",CPU_JOB_ID,"| inspect with: squeue -j",CPU_JOB_ID)

## 5 — Collect real execution evidence
This reads **only the log belonging to the current run**, not old screenshots or another student's output. **PENDING** means the job has not produced its completion marker yet. **FAIL** means the job terminated without completing, or numerical output is missing.

When the batch finishes, check the printed `MAP` lines and three `RESULT` lines. Each `RESULT` is generated by the compiled C heat solver on the compute node.

In [ ]:
CPU_RESULTS = {}
CPU_STATUS = "NOT RUN"
CPU_LOG = ""
if not CPU_JOB_ID:
    print("NOT RUN: no CPU batch job submitted.")
else:
    logfile=RUN_DIR / ("cpu-"+CPU_JOB_ID+".out")
    if not logfile.exists():
        CPU_STATUS="PENDING"
        print("PENDING: waiting for log",logfile.name)
    else:
        CPU_LOG=logfile.read_text(errors="replace")
        for line in CPU_LOG.splitlines():
            if line.startswith(("JOB ","MAP ","RESULT ","FAIL:","srun:","slurmstepd:")):print(line)
        for line in CPU_LOG.splitlines():
            if line.startswith("RESULT "):
                fields=dict(x.split("=",1) for x in line.split()[1:] if "=" in x)
                try:
                    mode=fields["mode"]
                    CPU_RESULTS[mode]={"ranks":int(fields["ranks"]),"threads":int(fields["threads"]),
                      "n":int(fields["n"]),"steps":int(fields["steps"]),"elapsed_s":float(fields["elapsed_s"]),
                      "checksum":float(fields["checksum"]),"probe":float(fields["probe"])}
                except (ValueError,KeyError) as e:print("FAIL: unreadable RESULT:",e)
        if "CPU_LAB_DONE" not in CPU_LOG: CPU_STATUS="PENDING" if shutil.which("squeue") and subprocess.run(["squeue","-h","-j",CPU_JOB_ID],capture_output=True,text=True,timeout=20).stdout.strip() else "FAIL"
        elif set(CPU_RESULTS)=={"serial","mpi","hybrid"} and all((RUN_DIR/(m+".bin")).exists() for m in CPU_RESULTS):
            # Full-field validation uses only Python stdlib, even if NumPy is absent.
            try:
                from array import array
                def read_grid(mode):
                    buf=array("d")
                    buf.frombytes((RUN_DIR/(mode+".bin")).read_bytes())
                    if len(buf)!=GRID_N*GRID_N:raise ValueError("Unexpected grid size: "+mode)
                    return buf
                reference=read_grid("serial")
                differences={mode:max(abs(a-b) for a,b in zip(reference,read_grid(mode)))
                             for mode in ("mpi","hybrid")}
                print("Maximum full-grid differences vs serial:", differences)
                summary_ok=all(abs(CPU_RESULTS[x]["checksum"]-CPU_RESULTS["serial"]["checksum"])<=1e-6
                               and abs(CPU_RESULTS[x]["probe"]-CPU_RESULTS["serial"]["probe"])<=1e-8
                               for x in ("mpi","hybrid"))
                CPU_STATUS="PASS" if summary_ok and all(err<=1e-9 for err in differences.values()) else "FAIL"
            except Exception as e:
                CPU_STATUS="FAIL"
                print("Full-grid check failed:",repr(e))
        else: CPU_STATUS="FAIL"
        if CPU_STATUS=="FAIL":print("\n--- Tail of actual batch log ---\n",CPU_LOG[-3500:])
    print("CPU status:",CPU_STATUS)
    if CPU_STATUS=="PENDING": print("Rerun THIS COLLECTION cell later. Do not resubmit another job.")

## 6 — Activity A: observe ranks and threads
**Problem:** Are MPI processes and OpenMP threads the same thing?

**What actually ran:** two MPI ranks, each using two OpenMP threads; inside a critical region each prints its rank and thread ID.

**Predict:** Will output contain `rank=0 thread=0` and `rank=1 thread=1`? Does output order need to be rank order?

**Observe:** four actual MAP lines (when CPU_STATUS is PASS), even if their order differs.

**Try:** compare the map to slide 12's 2×8-core placement options. A rank is a *process identity*; it is not necessarily a different physical node.

**Explain:** MPI uses separate process address spaces; OpenMP threads share an address space within one rank.

In [ ]:
if CPU_STATUS!="PASS":
    print("NOT RUN (this activity): first complete the CPU job. Current status:",CPU_STATUS)
else:
    m=[line for line in CPU_LOG.splitlines() if line.startswith("MAP ")]
    print("Observed MAP lines:",len(m))
    for line in sorted(m):print(line)
    distinct=set((re.search(r"rank=(\d+)",line).group(1),re.search(r"thread=(\d+)",line).group(1)) for line in m)
    print("MAP correctness:", "PASS" if len(m)==4 and len(distinct)==4 else "FAIL")

## 7 — Activity B: does MPI + OpenMP compute the same physics?
**Problem:** We can distribute work only if the final temperature field remains correct.

**What actually ran:** same five-point stencil for all three modes; ranks exchange halo rows at every step. On every time step the `old` input grid is separate from `next` output.

**Predict:** Should the checksums and probe at temperature grid cell (row 8, centre column) match?

**Observe:** checksums and probe values from the compute job, then compare **every grid value** using NumPy if available. Results are only marked PASS when actually checked.

**Try:** increase `STEPS` to 450, rerun **from setup**, and see how far heat has diffused.

In [ ]:
if CPU_STATUS!="PASS":
    print("NOT RUN: CPU results are required.")
else:
    print(f"{'mode':<12} {'ranks x threads':<18} {'checksum':>18} {'T[8,mid]':>14} {'seconds':>12}")
    for mode in ("serial","mpi","hybrid"):
        r=CPU_RESULTS[mode];print(f"{mode:<12} {r['ranks']} x {r['threads']:<14} {r['checksum']:>18.6f} {r['probe']:>14.6f} {r['elapsed_s']:>12.6f}")
    try:
        import numpy as np
        base=np.fromfile(RUN_DIR/"serial.bin",dtype=np.float64).reshape((GRID_N,GRID_N))
        for mode in ("mpi","hybrid"):
            other=np.fromfile(RUN_DIR/(mode+".bin"),dtype=np.float64).reshape((GRID_N,GRID_N))
            error=float(np.max(np.abs(base-other)))
            print(f"Full-grid verification for {mode}: max absolute difference = {error:.3g} ->", "PASS" if error<=1e-9 else "FAIL")
        print("Physical sanity (top boundary 100, bottom and non-top sides 0):",
              "PASS" if np.all(base[0,:]==100) and np.all(base[-1,:]==0) and np.all(base[1:,0]==0) and np.all(base[1:,-1]==0) else "FAIL")
    except ImportError:
        print("SKIPPED: NumPy not installed in kernel; compute-job checksum/probe cross-check still executed.")
    except Exception as e: print("FAIL: field verification error:",repr(e))

## 8 — Activity C: see the heat flow
**Situation:** A hot upper edge heats the plate. We expect temperature to be largest near the top and lower farther away.

**What actually ran:** the C MPI solver produced `serial.bin`, `mpi.bin` and `hybrid.bin` files containing real final temperatures. The notebook plots the binary array from the selected run. No heatmap is fabricated.

**Predict:** after more time steps, does the warm band move deeper into the plate?

**Try:** change `STEPS` from 350 to 450 and compare the heatmap and sample temperature. Keep `GRID_N` fixed to make the scientific comparison fair.

In [ ]:
if CPU_STATUS!="PASS":
    print("NOT RUN: heat grid not yet produced.")
else:
    try:
        import numpy as np
        import matplotlib.pyplot as plt
        heat=np.fromfile(RUN_DIR/"hybrid.bin",dtype=np.float64).reshape((GRID_N,GRID_N))
        fig,ax=plt.subplots(figsize=(7,4))
        im=ax.imshow(heat,origin="upper",cmap="coolwarm",vmin=0,vmax=100,interpolation="nearest")
        ax.scatter([GRID_N//2],[8],c="black",s=15)
        ax.set(title=f"Actual final heat field: {GRID_N}×{GRID_N}, {STEPS} steps",xlabel="Column",ylabel="Row")
        fig.colorbar(im,ax=ax,label="Temperature (°C)")
        plt.tight_layout();plt.show()
        print(f"Observed: T[8,{GRID_N//2}]={heat[8,GRID_N//2]:.5f} °C, min={heat.min():.5f}, max={heat.max():.5f}")
    except ImportError:print("SKIPPED: optional NumPy/Matplotlib plotting packages unavailable.")
    except Exception as e:print("FAIL:",repr(e))

## 9 — Activity D: is the hybrid run faster?
**Problem:** Process/thread creation, MPI messages and barriers are not free.

**What actually ran:** the C program used `MPI_Wtime` and measured the **slowest-rank elapsed time** for the update-and-halo loop. Compilation and binary output were excluded. Each mode uses different resources, so this is a classroom architecture comparison, not an isolated OpenMP scaling benchmark.

**Predict:** For this modest grid, might the MPI+OpenMP version be *slower* than serial? Why?

**Observe:** real durations and ratios. Run several times with `NEW_RUN=True` if you want to discuss variability. Do not assume a speedup in advance.

**Try:** double `GRID_N` (if your class allocation permits) while keeping `STEPS` unchanged; note this changes the scientific problem and should not be compared as the same-workload speedup.

In [ ]:
if CPU_STATUS!="PASS":
    print("NOT RUN: no measured CPU results.")
else:
    base=CPU_RESULTS["serial"]["elapsed_s"]
    for mode in ("serial","mpi","hybrid"):
        r=CPU_RESULTS[mode]
        print(f"{mode:>7}: actual elapsed {r['elapsed_s']:.6f} s; "
              f"ratio serial/mode = {base/r['elapsed_s']:.3f}x; "
              f"layout {r['ranks']} rank(s) × {r['threads']} thread(s)")
    print("Interpretation: these are individual measured runs. Ratios include MPI/OpenMP overhead and different core allocations.")
    try:
        import matplotlib.pyplot as plt
        labels=["serial","MPI only","MPI + OpenMP"]
        values=[CPU_RESULTS[m]["elapsed_s"] for m in ("serial","mpi","hybrid")]
        fig,ax=plt.subplots(figsize=(7,3));bars=ax.bar(labels,values)
        ax.set(ylabel="Measured elapsed seconds",title="Actual SciTech CPU execution (not illustrative)")
        for bar,v in zip(bars,values):ax.text(bar.get_x()+bar.get_width()/2,v,f"{v:.4f}s",ha="center",va="bottom")
        plt.tight_layout();plt.show()
    except ImportError:print("SKIPPED: Matplotlib not installed")

## 10 — Optional real GPU experiment (one allocated NVIDIA GPU)
**This is an optional extension, not part of the required CPU lab.** Change `RUN_GPU=True` in the setup cell, rerun this materialization and submit cell, and wait for a separate Slurm GPU allocation.

**What actually runs:** compiled CUDA **vector addition** on one GPU. The code copies two vectors to the device, launches a kernel, copies the result back and verifies all elements. CUDA events time only the kernel; a CPU wall clock measures the **end-to-end** operation including allocations and transfers. Do **not** equate kernel-only time with end-to-end application speedup.

**Prediction:** Which number is larger: kernel-only or end-to-end time? Why?

The GPU code uses the supported `nvhpc/25.7` module when available. If no GPU allocation or compiler is available, the experiment reports **SKIPPED or NOT RUN**, never a fake speedup. One GPU is enough; multi-GPU MPI/GPU execution is a later extension.

In [ ]:
GPU_DIR=WORKSPACE/("gpu-"+hashlib.sha256((SRC_CUDA+GPU_SCRIPT+BUILD).encode()).hexdigest()[:16]+("-"+uuid.uuid4().hex[:6] if GPU_NEW_RUN else ""))
GPU_JOB_FILE=GPU_DIR/"gpu_job_id.txt"
GPU_JOB_ID=None
if not RUN_GPU:
    print("SKIPPED (optional): RUN_GPU=False. CPU notebook remains complete without a GPU.")
else:
    GPU_DIR.mkdir(parents=True,exist_ok=True)
    for name,source in {"gpu_vector_add.cu":SRC_CUDA,"gpu.sbatch":GPU_SCRIPT}.items():
        target=GPU_DIR/name
        if target.exists() and target.read_text()!=source:raise RuntimeError("Frozen GPU source mismatch")
        if not target.exists():target.write_text(source)
    if GPU_JOB_FILE.exists() and not GPU_NEW_RUN:
        GPU_JOB_ID=GPU_JOB_FILE.read_text().strip()
        print("Reuse GPU job:",GPU_JOB_ID)
    elif not shutil.which("sbatch"):
        print("NOT RUN: sbatch unavailable.")
    else:
        env=os.environ.copy()
        for key in ("SLURM_MEM_PER_CPU","SLURM_MEM_PER_GPU","SLURM_MEM_PER_NODE"):env.pop(key,None)
        p=subprocess.run(["sbatch","--parsable","--chdir="+str(GPU_DIR),str(GPU_DIR/"gpu.sbatch")],
                         capture_output=True,text=True,env=env,timeout=35)
        if p.returncode:print("FAIL: GPU submission:",p.stderr.strip())
        else:
            GPU_JOB_ID=p.stdout.strip().split(";")[0];GPU_JOB_FILE.write_text(GPU_JOB_ID)
            print("Submitted optional GPU job:",GPU_JOB_ID)

## 11 — Collect optional GPU evidence
Inspect the real CUDA output:
- `GPU_RESULT` must include kernel and end-to-end milliseconds and a numerical error.
- **PASS** requires the job's completion marker and error <= 0.001.
- If compiler or runtime is unavailable the job prints `SKIPPED`.
- Merely detecting NVIDIA software on the Jupyter host does not prove you own a GPU.

This exercise does not claim GPU acceleration beats CPU until a properly matched CPU comparison is performed.

In [ ]:
GPU_STATUS="NOT RUN"
if not RUN_GPU: GPU_STATUS="SKIPPED"
elif not GPU_JOB_ID:GPU_STATUS="NOT RUN"
else:
    gpu_logfile=GPU_DIR/("gpu-"+GPU_JOB_ID+".out")
    if not gpu_logfile.exists():GPU_STATUS="PENDING"
    else:
        gpu_log=gpu_logfile.read_text(errors="replace")
        print(gpu_log[-3500:])
        if "GPU_LAB_DONE" in gpu_log and "GPU_RESULT" in gpu_log:
            m=re.search(r"max_error=([0-9.eE+-]+)",gpu_log)
            GPU_STATUS="PASS" if m and float(m.group(1))<=0.001 else "FAIL"
        elif "SKIPPED:" in gpu_log:GPU_STATUS="SKIPPED"
        else:
            active=shutil.which("squeue") and subprocess.run(["squeue","-h","-j",GPU_JOB_ID],capture_output=True,text=True,timeout=20).stdout.strip()
            GPU_STATUS="PENDING" if active else "FAIL"
print("Optional GPU status:",GPU_STATUS)
if GPU_STATUS=="PENDING":print("Rerun only this collection cell later.")

## 12 — [CHECKPOINT] Four short questions

**Q1.** Two CPU nodes each have eight cores. Is `2 MPI ranks × 8 threads` always faster than `4 ranks × 4 threads`?
- A. Yes, because it has fewer ranks.
- B. Yes, because OpenMP is always cheaper.
- C. No. Placement, workload and communication determine performance.
- D. No, because MPI cannot work within one node.

**Q2.** In the heat solver, what is the purpose of an MPI halo exchange?
- A. Copy the entire simulation into every rank.
- B. Supply neighbouring temperatures needed by a local stencil.
- C. Make all ranks use shared memory.
- D. Replace synchronization between time steps.

**Q3.** The GPU kernel takes 0.2 ms, but the complete GPU operation takes 4 ms. Which is justified?
- A. The entire GPU solution takes 0.2 ms.
- B. Data movement and setup contribute to total time.
- C. The GPU result must be incorrect.
- D. MPI automatically fixes the difference.

**Q4.** Which check best supports numerical correctness of hybrid heat computation?
- A. The hybrid version is the fastest.
- B. MPI prints two rank IDs.
- C. Compare the complete final grids with the same parameters.
- D. GPU utilization is 100%.

<details><summary>Answers (reveal after discussion)</summary>

- **Q1 C**: neither layout is universally best; A/B ignore resource placement and D is false.
- **Q2 B**: halo cells store copies of nearby rank-owned data; A/C/D misstate their function.
- **Q3 B**: end-to-end time includes costs outside the kernel; the smaller number is not whole-application latency.
- **Q4 C**: numerical agreement is independent of speed and resource utilization.

</details>

## 13 — One-change experiment and takeaway
Choose exactly one:
1. `STEPS = 450`, keeping the grid the same: **scientific change**. Predict the temperature near the top.
2. `GRID_N = 512`, same steps: **problem-size change**. Predict whether overhead becomes less important.
3. `NEW_RUN = True` without changing settings: **measurement repeat**. Discuss timing variability.
4. `RUN_GPU = True` (only when permitted): optional one-GPU experiment.

**Record:** prediction, changed setting, real Slurm job ID, PASS/FAIL status, observed data, explanation. Keep screenshots or outputs as your own evidence.

### Four essentials
**MPI:** splits data between processes. **OpenMP:** uses cooperating CPU threads within a process. **GPU:** accelerates appropriate kernels, subject to movement costs. **Performance:** must be measured while checking correctness.

**Bridge to syllabus Session 22:** once the overall architecture is clear, how can we coordinate communication, threading and data placement in a scalable production application?